

























# 00 · Baseline

Baseline adalah prediksi paling sederhana yang **harus dikalahkan** oleh setiap model. Model yang tidak lebih baik dari baseline belum berguna.

| Baseline | Cara menebak | Skenario |
|---|---|---|
| **1 · Rata-rata per kelompok bidang** | Rata-rata `peminat` data latih untuk kelompok bidang yang sama | `dengan_lag`, `tanpa_lag` |
| **2 · Peminat tahun lalu** | `prediksi = peminat_lag1` | `dengan_lag` |

Setiap baseline dijalankan untuk data gabungan (SNBP+SNBT), SNBP saja, dan SNBT saja. Semua hasil dicatat ke `hasil/evaluasi.csv` lewat `evaluasi()`.

In [1]:
import sys
from pathlib import Path

# Cari folder akar proyek (yang berisi folder src/), lalu daftarkan src/
AKAR = Path.cwd()
while not (AKAR / "src" / "fondasi.py").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "src"))

import pandas as pd
from fondasi import (muat_data, evaluasi, info_baris, lipatan_waktu,
                     ke_log, dari_log, SKENARIO, FILE_EVALUASI, RANDOM_STATE)

## Baseline 1 · Rata-rata peminat per kelompok bidang

Rata-rata dihitung **hanya dari data latih**, lalu diterapkan ke data uji 2025.

In [2]:
for skenario in SKENARIO:
    for jalur in (None, "SNBP", "SNBT"):
        X_train, X_test, y_train, y_test = muat_data(skenario, jalur, verbose=False)
        kelompok_latih = info_baris(X_train)["kelompok_bidang"].values
        kelompok_uji = info_baris(X_test)["kelompok_bidang"].values

        rata = y_train.groupby(kelompok_latih).mean()
        belum_ada = set(kelompok_uji) - set(rata.index)
        assert not belum_ada, f"Kelompok di data uji tidak ada di data latih: {belum_ada}"

        prediksi = pd.Series(kelompok_uji).map(rata).values
        evaluasi(y_test, prediksi, "baseline_rata_kelompok", skenario, jalur,
                 n_latih=len(X_train), n_fitur=1)
        print()

[evaluasi] baseline_rata_kelompok | dengan_lag | gabungan | target=asli
  MAE=304.55  MAPE=574.79% (0 baris peminat=0 dikecualikan)  R2=0.1402  RMSLE=1.4393

[evaluasi] baseline_rata_kelompok | dengan_lag | SNBP | target=asli
  MAE=219.57  MAPE=567.34% (0 baris peminat=0 dikecualikan)  R2=0.1405  RMSLE=1.4137



[evaluasi] baseline_rata_kelompok | dengan_lag | SNBT | target=asli
  MAE=368.15  MAPE=509.21% (0 baris peminat=0 dikecualikan)  R2=0.1692  RMSLE=1.4015



[evaluasi] baseline_rata_kelompok | tanpa_lag | gabungan | target=asli
  MAE=299.17  MAPE=608.94% (0 baris peminat=0 dikecualikan)  R2=0.1323  RMSLE=1.4569

[evaluasi] baseline_rata_kelompok | tanpa_lag | SNBP | target=asli
  MAE=215.25  MAPE=557.28% (0 baris peminat=0 dikecualikan)  R2=0.1390  RMSLE=1.4171



[evaluasi] baseline_rata_kelompok | tanpa_lag | SNBT | target=asli
  MAE=363.82  MAPE=611.87% (0 baris peminat=0 dikecualikan)  R2=0.1549  RMSLE=1.4388



## Baseline 2 · Peminat tahun lalu

Menebak bahwa peminat tahun ini sama dengan tahun lalu. Hanya bisa untuk skenario `dengan_lag`.

In [3]:
for jalur in (None, "SNBP", "SNBT"):
    X_train, X_test, y_train, y_test = muat_data("dengan_lag", jalur, verbose=False)
    prediksi = X_test["peminat_lag1"].values
    evaluasi(y_test, prediksi, "baseline_lag1", "dengan_lag", jalur,
             n_latih=len(X_train), n_fitur=1)
    print()

[evaluasi] baseline_lag1 | dengan_lag | gabungan | target=asli
  MAE=67.54  MAPE=24.33% (0 baris peminat=0 dikecualikan)  R2=0.9465  RMSLE=0.3746



[evaluasi] baseline_lag1 | dengan_lag | SNBP | target=asli
  MAE=48.96  MAPE=24.07% (0 baris peminat=0 dikecualikan)  R2=0.9400  RMSLE=0.3568

[evaluasi] baseline_lag1 | dengan_lag | SNBT | target=asli
  MAE=86.12  MAPE=24.59% (0 baris peminat=0 dikecualikan)  R2=0.9453  RMSLE=0.3916



## Ringkasan baseline

In [4]:
hasil = pd.read_csv(FILE_EVALUASI)
hasil = hasil[hasil["nama_model"].str.startswith("baseline")]
hasil[["nama_model", "skenario", "jalur", "n_latih", "n_uji", "MAE", "MAPE", "R2", "RMSLE"]] \
    .sort_values(["skenario", "jalur", "nama_model"]).reset_index(drop=True)

,nama_model,skenario,jalur,n_latih,n_uji,MAE,MAPE,R2,RMSLE
0,baseline_lag1,dengan_lag,SNBP,11729,4707,48.96,24.07,0.9400,0.3568
1,baseline_rata_kelompok,dengan_lag,SNBP,11729,4707,219.57,567.34,0.1405,1.4137
2,baseline_lag1,dengan_lag,SNBT,11716,4709,86.12,24.59,0.9453,0.3916
3,baseline_rata_kelompok,dengan_lag,SNBT,11716,4709,368.15,509.21,0.1692,1.4015
4,baseline_lag1,dengan_lag,gabungan,23445,9416,67.54,24.33,0.9465,0.3746
5,baseline_rata_kelompok,dengan_lag,gabungan,23445,9416,304.55,574.79,0.1402,1.4393
6,baseline_rata_kelompok,tanpa_lag,SNBP,16480,4890,215.25,557.28,0.1390,1.4171
7,baseline_rata_kelompok,tanpa_lag,SNBT,16479,4909,363.82,611.87,0.1549,1.4388
8,baseline_rata_kelompok,tanpa_lag,gabungan,32959,9799,299.17,608.94,0.1323,1.4569
